# 子图的 checkpointer 与中断传播

**场景：可复用的审批组件**。报销流程里"起草报销单 → 人工审批"是一段固定套路，
把它封装成子图后，任何父流程（报销、采购、出差……）插入这个节点就能获得审批能力，
审批细节对父图透明——这是子图内 human-in-the-loop 的真实卖点。

子图 `compile()` 时 checkpointer 有三种取值（默认不传即继承父图的）：

| checkpointer | 行为 |
|---|---|
| 不传 / `None` | **继承父图的 checkpointer**（默认） |
| `True` | 子图**独立持久化**，`get_state(config, subgraphs=True)` 能取到子图内部状态 |
| `False` | 子图不留自己的 checkpoint（实测 1.2.11：父图有 checkpointer 时，子图内 interrupt 仍能正常冒泡与恢复） |

子图内的 `interrupt()` 会**向上冒泡**到父图的 `__interrupt__`，`Command(resume=...)` 传入的值
再传回子图内 `interrupt()` 的返回值。中断基础见 [6_中断](../6_中断/) 章节。

## 用法一：审批子图直接作节点，interrupt 冒泡 + resume

In [5]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from rich import print
from typing_extensions import TypedDict


class ExpenseState(TypedDict):
    expense: str   # 报销事由
    draft: str     # 起草的报销单
    status: str    # 审批状态


# ---- 审批组件子图：起草 -> 人工审批（可插进任何父流程）----
def draft(state: ExpenseState):
    return {"draft": f"报销单：{state['expense']}，金额 500 元"}


def approve(state: ExpenseState):
    ok = interrupt(f"请审批：{state['draft']}（True/False）")
    return {"status": "已批准" if ok else "已驳回"}


expense_flow = (StateGraph(ExpenseState)
                .add_node("draft", draft)
                .add_node("approve", approve)
                .add_edge(START, "draft")
                .add_edge("draft", "approve")
                .add_edge("approve", END)
                .compile())   # 不传 checkpointer -> 继承父图的


# ---- 父流程：提交 -> 报销(子图) -> 归档 ----
def submit(state: ExpenseState):
    return {"expense": "出差高铁票"}


def archive(state: ExpenseState):
    return {"status": state["status"] + "，已归档"}


graph = (StateGraph(ExpenseState)
         .add_node("submit", submit)
         .add_node("expense_flow", expense_flow)
         .add_node("archive", archive)
         .add_edge(START, "submit")
         .add_edge("submit", "expense_flow")
         .add_edge("expense_flow", "archive")
         .compile(checkpointer=InMemorySaver()))

config = {"configurable": {"thread_id": "1"}}

# 第一次：提交报销申请（expense 由 submit 节点填写）；子图内的审批请求冒泡到顶层 __interrupt__
print(graph.invoke({}, config))

# 经理审批通过：值经父图传回子图内的 interrupt()，从审批点继续 -> 归档
print(graph.invoke(Command(resume=True), config))

{
    'expense': '出差高铁票',
    '__interrupt__': [
        Interrupt(
            value='请审批：报销单：出差高铁票，金额 500 元（True/False）',
            id='841e06548102e6b0d8d13c071e38770c'
        )
    ]
}

{'expense': '出差高铁票', 'draft': '报销单：出差高铁票，金额 500 元', 'status': '已批准，已归档'}

## 用法二：节点内调用审批子图，interrupt 同样冒泡 + resume

老系统里父流程逻辑都在普通函数库中时，审批子图在节点内 `invoke` 即可，
冒泡行为完全一致。注意：resume 后父图节点会**从头重跑**（官方文档明确行为），
中断点之前的代码会再执行一遍，副作用要放在 `interrupt()` 之后。

In [6]:
def reimburse(state: ExpenseState):
    return expense_flow.invoke(state)   # 审批子图当普通函数调用


graph2 = (StateGraph(ExpenseState)
          .add_node("submit", submit)
          .add_node("reimburse", reimburse)
          .add_node("archive", archive)
          .add_edge(START, "submit")
          .add_edge("submit", "reimburse")
          .add_edge("reimburse", "archive")
          .compile(checkpointer=InMemorySaver()))

config2 = {"configurable": {"thread_id": "2"}}
print(graph2.invoke({}, config2))
print(graph2.invoke(Command(resume=True), config2))

{
    'expense': '出差高铁票',
    '__interrupt__': [
        Interrupt(
            value='请审批：报销单：出差高铁票，金额 500 元（True/False）',
            id='57ea6ab4db3dfb67d1e20b9f9d3b8ad9'
        )
    ]
}

{'expense': '出差高铁票', 'draft': '报销单：出差高铁票，金额 500 元', 'status': '已批准，已归档'}

## 用法三：`checkpointer=True` 子图独立持久化

审计需求：审批卡在子图哪一步了？子图 `compile(checkpointer=True)` 后自带 checkpoint，
`get_state(config, subgraphs=True)` 的 `tasks[0].state` 非空，可拿到子图内部的
checkpoint config 做细粒度检查 / time-travel。

In [7]:
expense_flow3 = (StateGraph(ExpenseState)
                 .add_node("draft", draft)
                 .add_node("approve", approve)
                 .add_edge(START, "draft")
                 .add_edge("draft", "approve")
                 .add_edge("approve", END)
                 .compile(checkpointer=True))   # 独立持久化，隔离于父图

graph3 = (StateGraph(ExpenseState)
          .add_node("expense_flow", expense_flow3)
          .add_edge(START, "expense_flow")
          .compile(checkpointer=InMemorySaver()))

config3 = {"configurable": {"thread_id": "3"}}
graph3.invoke({"expense": "出差高铁票"}, config3)

# 审计：卡在子图哪一步？子图内部状态可查
snapshot = graph3.get_state(config3, subgraphs=True)
sub_snap = snapshot.tasks[0].state
print("审批进行到:", sub_snap.values["draft"], "| 等待节点:", sub_snap.next)

print(graph3.invoke(Command(resume=True), config3))

审批进行到: 报销单：出差高铁票，金额 500 元 | 等待节点:
('approve',)

{'expense': '出差高铁票', 'draft': '报销单：出差高铁票，金额 500 元', 'status': '已批准'}

## 实测结论（langgraph 1.2.11）

1. **interrupt 冒泡与 resume 对两种集成方式都生效**：直接作节点 / 节点内 `invoke`，审批请求都出现在顶层 `__interrupt__`，`Command(resume=...)` 都能把审批结果送回子图内的 `interrupt()`。
2. **checkpointer 默认继承父图**：子图 `compile()` 不传即可，interrupt / 多轮恢复直接可用。
3. **`checkpointer=True`**：子图独立 checkpoint，`get_state(subgraphs=True)` 的 `tasks[0].state` 非空，支持审计级细粒度访问。
4. **`checkpointer=False`**：子图不留自己的 checkpoint；父图有 checkpointer 时 interrupt 仍正常冒泡恢复（实测无报错）。
5. Interrupt 的 `id` 每次运行会变，匹配恢复值时用 id 对应，别写死。